# 🚀 Stack 2.9 - Colab Training Notebook

**Zero-cost training on Google Colab free tier with T4 GPU**

⏱️ **Expected runtime:** 3-5 hours
💾 **VRAM needed:** ~12GB (fits in T4's 15GB)

---

**CRITICAL:** Run cells in order from the top!

---

In [ ]:
# STEP 1: Setup - Mount Drive and define root directory
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT_DIR = "/content/drive/MyDrive/stack-2.9"
os.makedirs(ROOT_DIR, exist_ok=True)
os.chdir(ROOT_DIR)

print(f"✅ Working directory: {os.getcwd()}")
!ls -la

In [ ]:
# STEP 2: Clone repo (fresh every time)
import shutil

if os.path.exists('stack-2.9'):
    print("Removing old stack-2.9...")
    shutil.rmtree('stack-2.9')

!git clone https://github.com/my-ai-stack/stack-2.9.git

os.chdir(os.path.join(ROOT_DIR, 'stack-2.9'))
print(f"✅ In: {os.getcwd()}")
!ls -la

In [ ]:
# STEP 3: Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers peft accelerate datasets pyyaml tqdm scipy bitsandbytes
print("✅ Dependencies installed")

In [ ]:
# STEP 4: Download Base Model (Qwen2.5-Coder-7B)
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B"
MODEL_DIR = os.path.join(ROOT_DIR, "stack-2.9/base_model_qwen7b")

if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print(f"Downloading {MODEL_NAME} to {MODEL_DIR}...")
    print("This will take 15-20 minutes...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.save_pretrained(MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model.save_pretrained(MODEL_DIR)
    print(f"✅ Model saved")
else:
    print(f"✅ Model already exists")

!ls -lh {MODEL_DIR} | head -5

In [ ]:
# STEP 5: Find or download training data
REPO_DIR = os.path.join(ROOT_DIR, "stack-2.9")
DATA_PATH = None

# Check multiple possible locations
possible_paths = [
    os.path.join(REPO_DIR, "data/final/train.jsonl"),
    os.path.join(REPO_DIR, "training-data/final/train.jsonl"),
    os.path.join(REPO_DIR, "data_mini/train_mini.jsonl"),
]

for path in possible_paths:
    if os.path.exists(path):
        DATA_PATH = path
        print(f"✅ Found data at: {path}")
        break

# If not found, try to download from a URL or create small sample
if DATA_PATH is None:
    print("⚠️ Data not found in repo!")
    print("The training data (data/final/train.jsonl) is not in the GitHub repo.")
    print("Options:")
    print("  1. Upload train.jsonl to your Drive at: /content/drive/MyDrive/stack-2.9/data/final/train.jsonl")
    print("  2. Use a smaller dataset")
    
    # Create minimal sample data for testing (just 100 examples)
    print("\n📝 Creating minimal sample data (100 examples) for testing...")
    sample_data = []
    sample_prompt = """Write a Python function to reverse a string.
```python
def reverse_string(s):
    return s[::-1]
```"""
    sample_response = """Here's the function:
```python
def reverse_string(s):
    return s[::-1]
```
This uses Python slicing to reverse the string."""
    
    for i in range(100):
        sample_data.append({
            "messages": [
                {"role": "user", "content": sample_prompt},
                {"role": "assistant", "content": sample_response}
            ]
        })
    
    # Save sample
    import json
    sample_path = os.path.join(REPO_DIR, "data_mini/sample.jsonl")
    os.makedirs(os.path.dirname(sample_path), exist_ok=True)
    with open(sample_path, 'w') as f:
        for item in sample_data:
            f.write(json.dumps(item) + '\n')
    
    DATA_PATH = sample_path
    print(f"✅ Created sample data: {DATA_PATH}")

In [ ]:
# STEP 6: Prepare Training Configuration
import yaml

config_path = os.path.join(REPO_DIR, "stack/training/train_config_local.yaml")

if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config not found at: {config_path}")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update config with absolute paths
config['model']['name'] = MODEL_DIR
config['data']['input_path'] = DATA_PATH
OUTPUT_DIR = os.path.join(ROOT_DIR, "training_output")
config['output']['lora_dir'] = os.path.join(OUTPUT_DIR, "lora")
config['output']['merged_dir'] = os.path.join(OUTPUT_DIR, "merged")
config['hardware']['device'] = "cuda"
config['hardware']['num_gpus'] = 1

os.makedirs(OUTPUT_DIR, exist_ok=True)
updated_config_path = os.path.join(OUTPUT_DIR, "train_config.yaml")

with open(updated_config_path, 'w') as f:
    yaml.dump(config, f)

print(f"✅ Config saved to: {updated_config_path}")
print(f"   Model: {config['model']['name']}")
print(f"   Data: {config['data']['input_path']}")
print(f"   Device: {config['hardware']['device']}")

In [ ]:
# STEP 7: Train LoRA Adapter
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "stack/training"))

print("="*60)
print("STARTING TRAINING")
print("="*60)

from train_lora import train_lora
trainer = train_lora(updated_config_path)

print("="*60)
print("TRAINING COMPLETED")
print("="*60)

In [ ]:
# STEP 8: Verify and Merge
lora_dir = os.path.join(OUTPUT_DIR, "lora")
print(f"Checking LoRA: {lora_dir}")
if os.path.exists(lora_dir):
    !ls -lh {lora_dir}
else:
    print("❌ No LoRA output found")

In [ ]:
# STEP 9: Merge LoRA
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "stack/training"))
from merge_adapter import merge_adapter

merged_dir = os.path.join(OUTPUT_DIR, "merged")
os.makedirs(merged_dir, exist_ok=True)

merge_config = {
    'model': {'name': MODEL_DIR, 'trust_remote_code': True},
    'output': {'lora_dir': lora_dir, 'merged_dir': merged_dir},
    'quantization': {'enabled': False}
}

merge_config_path = os.path.join(OUTPUT_DIR, "merge_config.yaml")
with open(merge_config_path, 'w') as f:
    yaml.dump(merge_config, f)

merge_adapter(merge_config_path, lora_dir, merged_dir)
print(f"✅ Merged to: {merged_dir}")
!ls -lh {merged_dir}

## 🔚 Training Complete!

Your model is ready at:
`/content/drive/MyDrive/stack-2.9/training_output/merged/`

Download it from Google Drive!